# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [ ]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 88.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=ac4235b2-363f-45ad-8290-f4fa8546331f
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 91.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 28.1MB/s]


In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    adv_x = x.clone().detach()
    if rand_init:
        adv_x = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0, 1).detach()
    for i in range(steps):
        delta = adv_x - x
        grad = input_grad(model, x, delta, y)
        adv_x = adv_x + alpha * grad.sign()
        delta = adv_x - x
        delta = torch.clamp(delta, -eps, eps)
        adv_x = torch.clamp(x + delta, 0, 1).detach()
    return adv_x


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    adv_x = x.clone().detach()
    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp(min=1e-10)
        r = torch.rand(x.size(0), 1, 1, 1, device=x.device)
        adv_x = (x + clip_box(x, delta * eps * r)).detach()

    for i in range(steps):
        delta = adv_x - x
        grad = input_grad(model, x, delta, y)
        grad_norm = l2_norm(grad)
        adv_x = adv_x + alpha * grad / (grad_norm + 1e-10)
        delta = adv_x - x
        delta_norm = l2_norm(delta)
        factor = eps / delta_norm.clamp(min=1e-10)
        factor = torch.min(factor, torch.ones_like(factor))
        delta = delta * factor
        adv_x = (x + clip_box(x, delta)).detach()

    return adv_x

In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.2  l2 59.9  l1 22.8  union 22.8   (7 s)


{'clean': 78.7, 'linf': 48.2, 'l2': 59.9, 'l1': 22.8, 'union': 22.8}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [ ]:
def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)

In [ ]:
standard_model = adv_train(make_model(), standard_loss, epochs=5)
adv_model = adv_train(make_model(), single_norm_loss, epochs=5, norm="linf")

epoch 1: loss 1.896 (0 s)
epoch 2: loss 1.545 (1 s)
epoch 3: loss 1.380 (1 s)
epoch 4: loss 1.225 (2 s)
epoch 5: loss 1.134 (2 s)
epoch 1: loss 2.178 (3 s)
epoch 2: loss 2.093 (6 s)
epoch 3: loss 2.050 (8 s)
epoch 4: loss 2.021 (11 s)
epoch 5: loss 2.013 (14 s)


In [ ]:
standard_model.eval()
adv_model.eval()
with torch.no_grad():
    pred1 = standard_model(x_test).argmax(dim=1)
    pred2 = adv_model(x_test).argmax(dim=1)
    acc1 = (pred1 == y_test).float().mean().item() * 100
    acc2 = (pred2 == y_test).float().mean().item() * 100

print("standard accuracy:", acc1)
print("adversarial accuracy:", acc2)

standard accuracy: 58.70000123977661
adversarial accuracy: 37.700000405311584


In [ ]:
eps_values = [0.01, 0.02, 0.03, 0.04]
standard_model.eval()
adv_model.eval()
for eps in eps_values:
    x1 = pgd_linf(standard_model, x_test, y_test, eps, eps/4, 10)
    x2 = pgd_linf(adv_model, x_test, y_test, eps, eps/4, 10)

    with torch.no_grad():
        pred1 = standard_model(x1).argmax(dim=1)
        pred2 = adv_model(x2).argmax(dim=1)
        acc1 = (pred1 == y_test).float().mean().item() * 100
        acc2 = (pred2 == y_test).float().mean().item() * 100

    print("eps:", eps)
    print("standard accuracy:", round(acc1, 2))
    print("adversarial accuracy:", round(acc2, 2))
    print()

eps: 0.01
standard accuracy: 12.8
adversarial accuracy: 33.9

eps: 0.02
standard accuracy: 2.3
adversarial accuracy: 28.8

eps: 0.03
standard accuracy: 0.0
adversarial accuracy: 25.5

eps: 0.04
standard accuracy: 0.0
adversarial accuracy: 21.7



In [ ]:
eps_values = [0.01, 0.02, 0.03, 0.04]
standard_model.eval()
adv_model.eval()
for eps in eps_values:
    x1 = pgd_linf(standard_model, x_test, y_test, eps, eps, 1, rand_init=False)
    x2 = pgd_linf(adv_model, x_test, y_test, eps, eps, 1, rand_init=False)

    with torch.no_grad():
        pred1 = standard_model(x1).argmax(dim=1)
        pred2 = adv_model(x2).argmax(dim=1)
        acc1 = (pred1 == y_test).float().mean().item() * 100
        acc2 = (pred2 == y_test).float().mean().item() * 100

    print("FGSM eps:", eps)
    print("standard accuracy:", round(acc1, 2))
    print("adversarial accuracy:", round(acc2, 2))
    print()

FGSM eps: 0.01
standard accuracy: 18.2
adversarial accuracy: 33.8

FGSM eps: 0.02
standard accuracy: 6.6
adversarial accuracy: 29.0

FGSM eps: 0.03
standard accuracy: 2.9
adversarial accuracy: 25.8

FGSM eps: 0.04
standard accuracy: 1.3
adversarial accuracy: 22.5



# Problem 3 (a)

In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.200 (3 s)
epoch 2: loss 1.180 (6 s)
epoch 3: loss 1.154 (8 s)
epoch 4: loss 1.132 (11 s)
epoch 5: loss 1.098 (14 s)
epoch 6: loss 1.068 (17 s)
epoch 7: loss 1.043 (20 s)
epoch 8: loss 1.027 (23 s)
epoch 9: loss 1.011 (25 s)
epoch 10: loss 1.003 (28 s)
clean 78.8  linf 44.7  l2 57.1  l1 19.1  union 18.9   (7 s)


{'clean': 78.8, 'linf': 44.7, 'l2': 57.1, 'l1': 19.1, 'union': 18.9}

In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.896 (3 s)
epoch 2: loss 0.829 (6 s)
epoch 3: loss 0.784 (9 s)
epoch 4: loss 0.743 (11 s)
epoch 5: loss 0.709 (14 s)
epoch 6: loss 0.671 (17 s)
epoch 7: loss 0.629 (20 s)
epoch 8: loss 0.605 (23 s)
epoch 9: loss 0.587 (26 s)
epoch 10: loss 0.579 (29 s)
clean 81.9  linf 32.7  l2 60.4  l1 28.6  union 25.2   (7 s)


{'clean': 81.9, 'linf': 32.7, 'l2': 60.4, 'l1': 28.6, 'union': 25.2}

In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.227 (3 s)
epoch 2: loss 0.913 (6 s)
epoch 3: loss 0.650 (10 s)
epoch 4: loss 0.474 (13 s)
epoch 5: loss 0.335 (16 s)
epoch 6: loss 0.257 (19 s)
epoch 7: loss 0.222 (23 s)
epoch 8: loss 0.187 (26 s)
epoch 9: loss 0.159 (29 s)
epoch 10: loss 0.146 (32 s)
clean 82.7  linf 0.0  l2 0.1  l1 20.1  union 0.0   (7 s)


{'clean': 82.7, 'linf': 0.0, 'l2': 0.1, 'l1': 20.1, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q = model(x_q)
    logits_r = model(x_r)
    loss_q = F.cross_entropy(logits_q, y, reduction="none")
    loss_r = F.cross_entropy(logits_r, y, reduction="none")
    loss = torch.maximum(loss_q, loss_r).mean()
    correct = logits_q.argmax(1) == y

    if correct.sum() > 0:
        p = F.softmax(logits_q[correct], dim=1)
        logp = F.log_softmax(logits_q[correct], dim=1)
        logq = F.log_softmax(logits_r[correct], dim=1)
        kl = (p * (logp - logq)).sum(1).mean()
        loss = loss + lam * kl

    return loss


def gradient_projection(g_n, g_a):
    dot = (g_n * g_a).sum()
    norm_n = g_n.norm()
    norm_a = g_a.norm()
    cosine = dot / (norm_n * norm_a + 1e-12)

    if cosine > 0:
        return cosine * g_n

    return torch.zeros_like(g_n)

In [ ]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.430 (7 s)
epoch 2: loss 1.334 (13 s)
epoch 3: loss 1.300 (20 s)
epoch 4: loss 1.268 (27 s)
epoch 5: loss 1.253 (33 s)
epoch 6: loss 1.223 (40 s)
epoch 7: loss 1.200 (47 s)
epoch 8: loss 1.182 (53 s)
epoch 9: loss 1.174 (60 s)
epoch 10: loss 1.165 (67 s)
clean 78.5  linf 42.4  l2 62.0  l1 44.4  union 40.1   (7 s)


{'clean': 78.5, 'linf': 42.4, 'l2': 62.0, 'l1': 44.4, 'union': 40.1}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [ ]:
def my_loss(model, x, y, lam=0.1):
    model.eval()
    x_q = attack(model, x, y, "linf").detach()
    x_r = attack(model, x, y, "l1").detach()
    x_2 = attack(model, x, y, "l2").detach()
    model.train()

    x_q.requires_grad_(True)
    x_r.requires_grad_(True)
    logits_q = model(x_q)
    logits_r = model(x_r)
    logits_2 = model(x_2)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    ce_2 = F.cross_entropy(logits_2, y, reduction="none")
    robust_loss = torch.stack((ce_q, ce_r, ce_2), dim=0).max(dim=0).values.mean()


    g_q = torch.autograd.grad(ce_q.sum(), x_q, create_graph=True, retain_graph=True)[0]

    g_r = torch.autograd.grad(ce_r.sum(), x_r, retain_graph=True)[0].detach()

    similarity = F.cosine_similarity(g_q.flatten(1), g_r.flatten(1), dim=1)
    alignment_loss = (1.0 - similarity).mean()

    return robust_loss + lam * alignment_loss

In [ ]:
model_mine = adv_train(load_base_model(), my_loss)
test_model(model_mine)

epoch 1: loss 1.428 (11 s)
epoch 2: loss 1.324 (21 s)
epoch 3: loss 1.295 (32 s)
epoch 4: loss 1.269 (43 s)
epoch 5: loss 1.240 (53 s)
epoch 6: loss 1.211 (64 s)
epoch 7: loss 1.181 (75 s)
epoch 8: loss 1.165 (85 s)
epoch 9: loss 1.151 (96 s)
epoch 10: loss 1.144 (107 s)
clean 75.9  linf 43.3  l2 61.8  l1 47.1  union 41.7   (7 s)


{'clean': 75.9, 'linf': 43.3, 'l2': 61.8, 'l1': 47.1, 'union': 41.7}

In [ ]:
model_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
test_model(model_ramp_1)

model_mine_1 = adv_train(load_base_model(), my_loss, seed=1)
test_model(model_mine_1)

epoch 1: loss 1.433 (7 s)
epoch 2: loss 1.327 (13 s)
epoch 3: loss 1.296 (20 s)
epoch 4: loss 1.276 (27 s)
epoch 5: loss 1.253 (33 s)
epoch 6: loss 1.223 (40 s)
epoch 7: loss 1.205 (46 s)
epoch 8: loss 1.187 (53 s)
epoch 9: loss 1.177 (60 s)
epoch 10: loss 1.169 (66 s)
clean 78.7  linf 41.9  l2 61.9  l1 45.4  union 40.2   (7 s)
epoch 1: loss 1.430 (11 s)
epoch 2: loss 1.328 (21 s)
epoch 3: loss 1.292 (32 s)
epoch 4: loss 1.262 (42 s)
epoch 5: loss 1.234 (53 s)
epoch 6: loss 1.205 (64 s)
epoch 7: loss 1.182 (74 s)
epoch 8: loss 1.167 (85 s)
epoch 9: loss 1.155 (96 s)
epoch 10: loss 1.145 (106 s)
clean 76.3  linf 44.2  l2 62.3  l1 48.2  union 42.5   (7 s)


{'clean': 76.3, 'linf': 44.2, 'l2': 62.3, 'l1': 48.2, 'union': 42.5}

In [ ]:
test_model(model_mine_1, steps=200, restarts=5)

clean 76.3  linf 44.0  l2 62.3  l1 46.9  union 41.8   (68 s)


{'clean': 76.3, 'linf': 44.0, 'l2': 62.3, 'l1': 46.9, 'union': 41.8}

In [ ]:
def ablation_loss(model, x, y):
    model.eval()
    x_q = attack(model, x, y, "linf").detach()
    x_r = attack(model, x, y, "l1").detach()
    x_2 = attack(model, x, y, "l2").detach()
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)
    logits_2 = model(x_2)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    ce_2 = F.cross_entropy(logits_2, y, reduction="none")
    return torch.stack((ce_q, ce_r, ce_2), dim=0).max(dim=0).values.mean()


In [ ]:
model_ablation = adv_train(load_base_model(), ablation_loss, seed=0)
ablation_results = test_model(model_ablation)
print("Ablation union:", round(ablation_results["union"], 1))
print("Alignment model seed 0:", 41.6)


epoch 1: loss 1.403 (9 s)
epoch 2: loss 1.299 (18 s)
epoch 3: loss 1.270 (26 s)
epoch 4: loss 1.243 (35 s)
epoch 5: loss 1.212 (44 s)
epoch 6: loss 1.184 (52 s)
epoch 7: loss 1.153 (61 s)
epoch 8: loss 1.136 (70 s)
epoch 9: loss 1.121 (79 s)
epoch 10: loss 1.115 (87 s)
clean 76.4  linf 42.9  l2 61.6  l1 47.3  union 41.2   (7 s)
Ablation union: 41.2
Alignment model seed 0: 41.6


In [ ]:
model_ramp_equal = adv_train(
    load_base_model(), ramp_loss,
    epochs=16, gp_beta=0.5, seed=0
)

test_model(model_ramp_equal)

epoch 1: loss 1.430 (7 s)
epoch 2: loss 1.336 (13 s)
epoch 3: loss 1.302 (20 s)
epoch 4: loss 1.275 (27 s)
epoch 5: loss 1.264 (33 s)
epoch 6: loss 1.235 (40 s)
epoch 7: loss 1.209 (46 s)
epoch 8: loss 1.187 (53 s)
epoch 9: loss 1.169 (60 s)
epoch 10: loss 1.149 (66 s)
epoch 11: loss 1.136 (73 s)
epoch 12: loss 1.116 (80 s)
epoch 13: loss 1.106 (86 s)
epoch 14: loss 1.095 (93 s)
epoch 15: loss 1.093 (100 s)
epoch 16: loss 1.086 (106 s)
clean 78.4  linf 40.9  l2 60.1  l1 45.1  union 39.3   (7 s)


{'clean': 78.4, 'linf': 40.9, 'l2': 60.1, 'l1': 45.1, 'union': 39.3}